In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score
import warnings

warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train_df = pd.read_csv('C:\\Users\\USER\\Downloads\\DigiCow\\data\\raw\\Train.csv')
test_df = pd.read_csv('C:\\Users\\USER\\Downloads\\DigiCow\\data\\raw\\Test.csv')
prior_df = pd.read_csv('C:\\Users\\USER\\Downloads\\DigiCow\\data\\raw\\Prior.csv')

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()
prior_df.columns = prior_df.columns.str.strip()

# ====================================================
# PURE MULTI-HORIZON TARGET ENCODING
# ====================================================
print("Extracting Pure Multi-Horizon Truths...")

farmer_prior = prior_df.groupby('farmer_name').agg(
    farmer_prior_count=('ID', 'count'),
    farmer_07_rate=('adopted_within_07_days', 'mean'),
    farmer_90_rate=('adopted_within_90_days', 'mean'),
    farmer_120_rate=('adopted_within_120_days', 'mean')
).reset_index()

trainer_prior = prior_df.groupby('trainer').agg(
    trainer_07_rate=('adopted_within_07_days', 'mean'),
    trainer_90_rate=('adopted_within_90_days', 'mean'),
    trainer_120_rate=('adopted_within_120_days', 'mean')
).reset_index()

group_prior = prior_df.groupby('group_name').agg(
    group_07_rate=('adopted_within_07_days', 'mean'),
    group_90_rate=('adopted_within_90_days', 'mean'),
    group_120_rate=('adopted_within_120_days', 'mean')
).reset_index()

train_df['is_train'] = 1
test_df['is_train'] = 0
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

df = df.merge(farmer_prior, on='farmer_name', how='left')
df = df.merge(trainer_prior, on='trainer', how='left')
df = df.merge(group_prior, on='group_name', how='left')

G_07 = prior_df['adopted_within_07_days'].mean()
G_90 = prior_df['adopted_within_90_days'].mean()
G_120 = prior_df['adopted_within_120_days'].mean()

df['farmer_prior_count'] = df['farmer_prior_count'].fillna(0)
for horizon, G_mean in zip(['07', '90', '120'], [G_07, G_90, G_120]):
    df[f'farmer_{horizon}_rate'] = df[f'farmer_{horizon}_rate'].fillna(G_mean)
    df[f'trainer_{horizon}_rate'] = df[f'trainer_{horizon}_rate'].fillna(G_mean)
    df[f'group_{horizon}_rate'] = df[f'group_{horizon}_rate'].fillna(G_mean)

# ====================================================
# HIGH-RESOLUTION NLP (50 Components) & RESCUED BEHAVIOR
# ====================================================
# Safely extract Day of Week before dropping the date!
if 'training_day' in df.columns:
    df['training_day_temp'] = pd.to_datetime(df['training_day'], errors='coerce')
    df['training_day_of_week'] = df['training_day_temp'].dt.dayofweek.fillna(-1)
    df.drop(columns=['training_day_temp'], inplace=True)

topic_col = 'topics_list' if 'topics_list' in df.columns else 'topics'
if topic_col in df.columns:
    df[topic_col] = df[topic_col].fillna('unknown').astype(str).str.lower()
    df['topic_count'] = df[topic_col].apply(lambda x: len(x.split(',')))
    
    tfidf = TfidfVectorizer(max_features=50, analyzer='word', token_pattern=r'\w+')
    tfidf_mat = tfidf.fit_transform(df[topic_col])
    tfidf_df = pd.DataFrame(tfidf_mat.toarray(), columns=[f'tfidf_{i}' for i in range(50)])
    df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

# --- Safe Label Encoding ---
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in ['ID', 'farmer_name', 'topics', 'topics_list']:
    if col in cat_cols: cat_cols.remove(col)

for col in cat_cols:
    df[col] = df[col].astype(str).fillna('missing')
    df[col] = LabelEncoder().fit_transform(df[col])

# --- Splitting & Leakage Prevention ---
train_clean = df[df['is_train'] == 1].copy()
test_clean = df[df['is_train'] == 0].copy()

train_groups = train_clean['farmer_name'].values 

# THE FIX: Explicitly ignore 'training_day_of_week' from the drop list!
date_cols = [c for c in train_clean.columns if ('day' in c.lower() or 'date' in c.lower()) and c not in ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days', 'training_day_of_week']]

targets = ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days']
leakage_cols = ['ID', 'farmer_name', 'is_train', 'topics', 'topics_list', 'has_second_training'] + targets + date_cols

X = train_clean.drop(columns=[c for c in leakage_cols if c in train_clean.columns])
X_test = test_clean.drop(columns=[c for c in leakage_cols if c in test_clean.columns])
X = X.fillna(-999)
X_test = X_test.fillna(-999)

print(f"Rescued Behavioral Features ready! Final X shape: {X.shape}")

1. Loading datasets...
Extracting Pure Multi-Horizon Truths...
Rescued Behavioral Features ready! Final X shape: (13536, 72)


In [2]:
def train_15fold_unthinkable_blend(target_name, X_train, y_train_series, X_test_full, groups):
    print(f"\n--- Training 15-Fold Unthinkable Blend for {target_name} ---")
    oof_preds = np.zeros(len(X_train))
    test_preds = np.zeros(len(X_test_full))
    
    # THE 15-FOLD MASTER SQUEEZE
    sgkf = StratifiedGroupKFold(n_splits=15, shuffle=True, random_state=42)
    fold_aucs, fold_lls = [], []
    
    SEEDS = [42, 2024, 888] 
    
    for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_train, y_train_series, groups=groups)):
        X_trn, y_trn = X_train.iloc[trn_idx], y_train_series.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train_series.iloc[val_idx]
        
        fold_val_preds = np.zeros(len(X_val))
        fold_test_preds = np.zeros(len(X_test_full))
        
        for seed in SEEDS:
            # 1. LIGHTGBM (The Scalpel)
            lgb_base = lgb.LGBMClassifier(
                n_estimators=1500, learning_rate=0.01, max_depth=4,
                min_child_samples=10, scale_pos_weight=15.0, 
                subsample=0.85, colsample_bytree=0.4, 
                random_state=seed + fold, n_jobs=-1, verbose=-1
            )
            lgb_base.fit(X_trn, y_trn)
            val_lgb = lgb_base.predict_proba(X_val)[:, 1]
            test_lgb = lgb_base.predict_proba(X_test_full)[:, 1]
            
            iso_lgb = IsotonicRegression(out_of_bounds='clip').fit(val_lgb, y_val)
            cal_val_lgb = np.clip(iso_lgb.predict(val_lgb), 1e-15, 1)
            cal_test_lgb = np.clip(iso_lgb.predict(test_lgb), 1e-15, 1)
            
            # 2. CATBOOST (The Heavyweight)
            cb_base = CatBoostClassifier(
                iterations=1500, learning_rate=0.01, depth=5,
                scale_pos_weight=15.0, 
                colsample_bylevel=0.4, 
                verbose=0, random_seed=seed + fold
            )
            cb_base.fit(X_trn, y_trn)
            val_cb = cb_base.predict_proba(X_val)[:, 1]
            test_cb = cb_base.predict_proba(X_test_full)[:, 1]
            
            iso_cb = IsotonicRegression(out_of_bounds='clip').fit(val_cb, y_val)
            cal_val_cb = np.clip(iso_cb.predict(val_cb), 1e-15, 1)
            cal_test_cb = np.clip(iso_cb.predict(test_cb), 1e-15, 1)

            # THE ARITHMETIC BLEND (40% LGB, 60% CB)
            arithmetic_val = (cal_val_lgb * 0.40) + (cal_val_cb * 0.60)
            arithmetic_test = (cal_test_lgb * 0.40) + (cal_test_cb * 0.60)
            
            # THE GEOMETRIC BLEND (40% LGB, 60% CB)
            geometric_val = (cal_val_lgb ** 0.40) * (cal_val_cb ** 0.60)
            geometric_test = (cal_test_lgb ** 0.40) * (cal_test_cb ** 0.60)
            
            # THE 70/30 HYBRID MASTER BLEND
            seed_val_blend = (geometric_val * 0.70) + (arithmetic_val * 0.30)
            seed_test_blend = (geometric_test * 0.70) + (arithmetic_test * 0.30)
            
            fold_val_preds += seed_val_blend / len(SEEDS)
            fold_test_preds += seed_test_blend / len(SEEDS)
            
        fold_val_preds = np.clip(fold_val_preds, 1e-15, 1 - 1e-15)
        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / 15 # Divided by 15 for 15 folds!
        
        auc = roc_auc_score(y_val, fold_val_preds)
        ll = log_loss(y_val, fold_val_preds)
        fold_aucs.append(auc)
        fold_lls.append(ll)
        print(f"Fold {fold+1:02d} | AUC: {auc:.5f} | LogLoss: {ll:.5f}")
        
    print(f">> Final BLEND {target_name} | MEAN AUC: {np.mean(fold_aucs):.5f} | MEAN LogLoss: {np.mean(fold_lls):.5f}")
    return test_preds

In [3]:
# ====================================================
# PARALLEL EXECUTION 
# ====================================================
print("\nGenerating 15-Fold Rescued predictions...")
pred_07 = train_15fold_unthinkable_blend('adopted_within_07_days', X, train_clean['adopted_within_07_days'], X_test, train_groups)
pred_90 = train_15fold_unthinkable_blend('adopted_within_90_days', X, train_clean['adopted_within_90_days'], X_test, train_groups)
pred_120 = train_15fold_unthinkable_blend('adopted_within_120_days', X, train_clean['adopted_within_120_days'], X_test, train_groups)

# ====================================================
# MONOTONICITY & THE 0.01% DAMPENER
# ====================================================
print("\nEnforcing Monotonicity and Applying the 0.01% Dampener...")
final_90 = np.maximum(pred_90, pred_07)
final_120 = np.maximum(pred_120, final_90)

sub = test_df[['ID']].copy()

sub['Target_07_AUC'] = pred_07
sub['Target_07_LogLoss'] = pred_07
sub['Target_90_AUC'] = final_90
sub['Target_90_LogLoss'] = final_90
sub['Target_120_AUC'] = final_120
sub['Target_120_LogLoss'] = final_120

# THE UNTHINKABLE DAMPENER
for col in sub.columns:
    if col != 'ID':
        sub[col] = (sub[col] * 0.999) + 0.0005

file_name = 'Sub_Final_15Fold_Rescued.csv'
sub.to_csv(file_name, index=False)
print(f"\nBOOM! The 15-Fold Master Squeeze is complete. Submission saved to {file_name}.")


Generating 15-Fold Rescued predictions...

--- Training 15-Fold Unthinkable Blend for adopted_within_07_days ---
Fold 01 | AUC: 0.99563 | LogLoss: 0.01358
Fold 02 | AUC: 0.99503 | LogLoss: 0.01427
Fold 03 | AUC: 0.98358 | LogLoss: 0.02596
Fold 04 | AUC: 0.99464 | LogLoss: 0.01928
Fold 05 | AUC: 0.99574 | LogLoss: 0.01484
Fold 06 | AUC: 0.99417 | LogLoss: 0.01320
Fold 07 | AUC: 0.98862 | LogLoss: 0.02167
Fold 08 | AUC: 0.99565 | LogLoss: 0.01397
Fold 09 | AUC: 0.99719 | LogLoss: 0.01326
Fold 10 | AUC: 0.99742 | LogLoss: 0.01256
Fold 11 | AUC: 0.99300 | LogLoss: 0.01950
Fold 12 | AUC: 0.98955 | LogLoss: 0.01831
Fold 13 | AUC: 0.99537 | LogLoss: 0.01016
Fold 14 | AUC: 0.97598 | LogLoss: 0.02245
Fold 15 | AUC: 0.99556 | LogLoss: 0.01592
>> Final BLEND adopted_within_07_days | MEAN AUC: 0.99247 | MEAN LogLoss: 0.01660

--- Training 15-Fold Unthinkable Blend for adopted_within_90_days ---
Fold 01 | AUC: 0.98861 | LogLoss: 0.01878
Fold 02 | AUC: 0.98915 | LogLoss: 0.02460
Fold 03 | AUC: 0.98